# UCU - Proyecto Webscraping
Proyecto para la Licenciatura en Datos y Negocios de la Universidad Católica del Uruguay

In [13]:
import requests
from bs4 import BeautifulSoup
import json

In [14]:
import json
import re
import requests
from bs4 import BeautifulSoup

In [15]:
## Definir las ciudades
ciudades = {
    "Montevideo": "https://www.infocasas.com.uy/venta/casas/montevideo",
    "Ciudad de la Costa": "https://m.infocasas.com.uy/venta/casas/canelones/ciudad-de-la-costa",
    "Colonia del Sacramento": "https://www.infocasas.com.uy/venta/casas/colonia/colonia-del-sacramento"
}

In [16]:
def limpiar_precio(texto):
    resultado = re.search(r"U\$S\s?([\d\.]+)", texto)

    if resultado:
        precio = resultado.group(1)
        precio = precio.replace(".", "")
        return int(precio)

    return None


def obtener_numero(texto, patron):
    resultado = re.search(patron, texto, re.IGNORECASE)

    if resultado:
        return int(resultado.group(1))

    return None

In [17]:
## Scrapping de las ciudades

def scrapear_ciudad(nombre_ciudad, url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    respuesta = requests.get(url, headers=headers)
    soup = BeautifulSoup(respuesta.text, "html.parser")

    propiedades = []

    links = soup.find_all("a", href=True)

    for link in links:
        texto = link.get_text(" ", strip=True)
        href = link["href"]

        if "U$S" in texto and len(propiedades) < 10:
            precio = limpiar_precio(texto)
            habitaciones = obtener_numero(texto, r"(\d+)\s*Dorm")
            tamano = obtener_numero(texto, r"(\d+)\s*m²")

            if href.startswith("/"):
                href = "https://www.infocasas.com.uy" + href

            propiedad = {
                "precio": precio,
                "tamano": tamano,
                "habitaciones": habitaciones,
                "link": href
            }

            propiedades.append(propiedad)

    return {
        "nombre": nombre_ciudad,
        "propiedades": propiedades
    }

In [18]:
## Revisar los datos de cada ciudad

datos = {
    "ciudades": []
}

for nombre_ciudad, url in ciudades.items():
    ciudad = scrapear_ciudad(nombre_ciudad, url)
    datos["ciudades"].append(ciudad)

In [19]:
## Guardar en JSON

with open("propiedades.json", "w", encoding="utf-8") as archivo:
    json.dump(datos, archivo, indent=4, ensure_ascii=False)

print("Archivo JSON creado correctamente.")

Archivo JSON creado correctamente.


|Utilice la web de infocasas para realizar la consulta. Elegi Montevideo, Ciudad de la costa y Colonía del sacramento porque son ciudades que puede haber alta oferta de venta de casas